# BÀI TẬP 4 – Trực quan hóa Dữ liệu Không gian & Dashboard Tương tác

**Sinh viên:** Lê Huỳnh Anh Khôi  
**MSSV:** 24133034  
**Chương:** 3  

---
## Bước 1: Thu thập / Định nghĩa Dữ liệu Không gian

In [ ]:
# Cài đặt thư viện nếu chưa có
import subprocess, sys
pkgs = ['pandas', 'plotly', 'ipywidgets']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ Tất cả thư viện đã sẵn sàng!')

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

# =====================================================
# ĐỊNH NGHĨA DATAFRAME – 10 Tỉnh/Thành phố Việt Nam
# =====================================================
np.random.seed(42)

data = {
    'TinhThanh': [
        'Hà Nội', 'Hải Phòng', 'Quảng Ninh',
        'Đà Nẵng', 'Huế', 'Quảng Nam',
        'TP. Hồ Chí Minh', 'Bình Dương', 'Đồng Nai', 'Cần Thơ'
    ],
    'KinhDo': [
        105.8342, 106.6881, 107.0765,
        108.2022, 107.5850, 108.0120,
        106.6297, 106.6524, 107.0738, 105.7469
    ],
    'ViDo': [
        21.0278,  20.8449,  21.0064,
        16.0544,  16.4637,  15.8794,
        10.8231,  11.0686,  10.9574,  10.0452
    ],
    'DoanhThu': [
        4200, 1850, 1420,
        2100, 980,  760,
        6800, 3100, 2450, 1600
    ],
    'SoChiNhanh': [
        85, 42, 31,
        58, 24, 18,
        130, 67, 54, 39
    ],
    'VungMien': [
        'Bắc', 'Bắc', 'Bắc',
        'Trung', 'Trung', 'Trung',
        'Nam', 'Nam', 'Nam', 'Nam'
    ]
}

df = pd.DataFrame(data)

print('=' * 55)
print('       DATAFRAME – 10 TỈNH/THÀNH PHỐ VIỆT NAM')
print('=' * 55)
display(df.style
    .set_caption('Dữ liệu kinh doanh theo địa lý')
    .background_gradient(subset=['DoanhThu'], cmap='YlOrRd')
    .background_gradient(subset=['SoChiNhanh'], cmap='Blues')
    .format({'DoanhThu': '{:,} tỷ', 'KinhDo': '{:.4f}°', 'ViDo': '{:.4f}°'})
)

print(f'\n📊 Tổng số tỉnh/thành phố: {len(df)}')
print(f'📈 Tổng doanh thu: {df["DoanhThu"].sum():,} tỷ đồng')
print(f'🏢 Tổng số chi nhánh: {df["SoChiNhanh"].sum()}')
print(f'🗺️  Vùng miền: {df["VungMien"].value_counts().to_dict()}')

---
## Bước 2: Tác vụ Trực quan hóa & Tích hợp

### Task 4.1 – Geospatial Bubble Map
Sử dụng `plotly.express.scatter_geo`:  
- **Kích thước bong bóng** → `DoanhThu`  
- **Màu sắc** → `VungMien`

In [ ]:
# =====================================================
# TASK 4.1 – GEOSPATIAL BUBBLE MAP
# =====================================================

color_map = {
    'Bắc':   '#1f77b4',
    'Trung':  '#ff7f0e',
    'Nam':   '#2ca02c'
}

fig_map = px.scatter_geo(
    df,
    lat='ViDo',
    lon='KinhDo',
    size='DoanhThu',
    color='VungMien',
    color_discrete_map=color_map,
    hover_name='TinhThanh',
    hover_data={
        'DoanhThu': ':,',
        'SoChiNhanh': True,
        'KinhDo': ':.4f',
        'ViDo': ':.4f'
    },
    size_max=55,
    projection='natural earth',
    title='🗺️ Task 4.1 – Bản đồ Bong bóng Doanh thu theo Tỉnh/Thành phố Việt Nam',
    labels={'VungMien': 'Vùng Miền', 'DoanhThu': 'Doanh Thu (tỷ đồng)'}
)

fig_map.update_geos(
    visible=True,
    resolution=50,
    showcountries=True,
    countrycolor='gray',
    showcoastlines=True,
    coastlinecolor='steelblue',
    showland=True,
    landcolor='#f5f5dc',
    showocean=True,
    oceancolor='#cce5ff',
    showlakes=True,
    lakecolor='#cce5ff',
    lonaxis_range=[100, 115],
    lataxis_range=[8, 24]
)

fig_map.update_layout(
    title_font_size=16,
    title_font_color='#333',
    title_x=0.5,
    legend_title_text='Vùng Miền',
    legend=dict(orientation='v', x=1.02, y=0.5),
    height=600,
    paper_bgcolor='white',
    margin=dict(l=0, r=150, t=60, b=0)
)

# Thêm nhãn tên tỉnh/thành
for _, row in df.iterrows():
    fig_map.add_trace(go.Scattergeo(
        lat=[row['ViDo'] + 0.35],
        lon=[row['KinhDo']],
        mode='text',
        text=[row['TinhThanh']],
        textfont=dict(size=9, color='#333'),
        showlegend=False,
        hoverinfo='skip'
    ))

fig_map.show()
print('✅ Task 4.1 hoàn thành – Geospatial Bubble Map')

---
### Task 4.2 – Mini Interactive Dashboard
Dashboard gồm **2 biểu đồ**:
1. **Bản đồ không gian** (Task 4.1) – lọc theo vùng miền
2. **Biểu đồ Cột** doanh thu theo từng tỉnh/thành của vùng được chọn

Sử dụng `ipywidgets.Dropdown` để chọn vùng miền lọc dữ liệu.

In [ ]:
# =====================================================
# TASK 4.2 – MINI INTERACTIVE DASHBOARD
# =====================================================

color_map = {
    'Bắc':   '#1f77b4',
    'Trung':  '#ff7f0e',
    'Nam':   '#2ca02c'
}

dropdown = widgets.Dropdown(
    options=['Tất cả', 'Bắc', 'Trung', 'Nam'],
    value='Tất cả',
    description='🗺️ Vùng Miền:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)

output = widgets.Output()


def build_dashboard(vung_mien):
    if vung_mien == 'Tất cả':
        df_filtered = df.copy()
        title_suffix = 'Tất cả Vùng Miền'
    else:
        df_filtered = df[df['VungMien'] == vung_mien].copy()
        title_suffix = f'Vùng {vung_mien}'

    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.55, 0.45],
        specs=[[{'type': 'scattergeo'}, {'type': 'bar'}]],
        subplot_titles=[
            '🗺️ Bản đồ Bong bóng – Kích thước = Doanh Thu',
            '📊 Doanh Thu theo Tỉnh/Thành phố'
        ]
    )

    # --- Bản đồ ---
    for vung in df_filtered['VungMien'].unique():
        df_v = df_filtered[df_filtered['VungMien'] == vung]
        fig.add_trace(
            go.Scattergeo(
                lat=df_v['ViDo'],
                lon=df_v['KinhDo'],
                mode='markers+text',
                marker=dict(
                    size=df_v['DoanhThu'] / 80,
                    color=color_map[vung],
                    opacity=0.75,
                    line=dict(color='white', width=1.5)
                ),
                text=df_v['TinhThanh'],
                textposition='top center',
                textfont=dict(size=9),
                name=f'Vùng {vung}',
                customdata=df_v[['DoanhThu', 'SoChiNhanh']].values,
                hovertemplate=(
                    '<b>%{text}</b><br>'
                    'Doanh Thu: %{customdata[0]:,} tỷ<br>'
                    'Số Chi Nhánh: %{customdata[1]}<br>'
                    '<extra></extra>'
                )
            ),
            row=1, col=1
        )

    # --- Biểu đồ Cột ---
    df_sorted = df_filtered.sort_values('DoanhThu', ascending=True)
    bar_colors = [color_map[v] for v in df_sorted['VungMien']]

    fig.add_trace(
        go.Bar(
            x=df_sorted['DoanhThu'],
            y=df_sorted['TinhThanh'],
            orientation='h',
            marker_color=bar_colors,
            marker_line=dict(color='white', width=0.8),
            text=[f'{v:,} tỷ' for v in df_sorted['DoanhThu']],
            textposition='outside',
            textfont=dict(size=10),
            name='Doanh Thu',
            showlegend=False,
            hovertemplate=(
                '<b>%{y}</b><br>'
                'Doanh Thu: %{x:,} tỷ đồng<br>'
                '<extra></extra>'
            )
        ),
        row=1, col=2
    )

    fig.update_geos(
        visible=True,
        resolution=50,
        showcountries=True,
        countrycolor='#aaa',
        showcoastlines=True,
        coastlinecolor='steelblue',
        showland=True,
        landcolor='#f5f5dc',
        showocean=True,
        oceancolor='#d0e8ff',
        lonaxis_range=[100, 115],
        lataxis_range=[8, 24]
    )

    fig.update_xaxes(title_text='Doanh Thu (tỷ đồng)', row=1, col=2, showgrid=True, gridcolor='#eee')
    fig.update_yaxes(title_text='Tỉnh/Thành phố', row=1, col=2)

    tong_dt = df_filtered['DoanhThu'].sum()
    tong_cn = df_filtered['SoChiNhanh'].sum()

    fig.update_layout(
        title=dict(
            text=(
                f'📊 DASHBOARD KINH DOANH – {title_suffix}  |  '
                f'Tổng DT: {tong_dt:,} tỷ  |  Tổng CN: {tong_cn}'
            ),
            x=0.5,
            font=dict(size=13, color='#222')
        ),
        height=520,
        paper_bgcolor='#fafafa',
        plot_bgcolor='white',
        legend=dict(title='Vùng Miền', orientation='h', x=0.0, y=-0.08),
        margin=dict(l=20, r=20, t=80, b=60)
    )
    return fig


def on_change(change):
    if change['name'] == 'value':
        with output:
            from IPython.display import clear_output
            clear_output(wait=True)
            build_dashboard(change['new']).show()


dropdown.observe(on_change, names='value')

header = widgets.HTML(value='''
<div style="background:linear-gradient(135deg,#667eea,#764ba2);color:white;
            padding:16px 24px;border-radius:10px;margin-bottom:12px;font-family:Arial,sans-serif;">
  <h2 style="margin:0;font-size:20px;">🚀 Task 4.2 – Mini Interactive Dashboard</h2>
  <p style="margin:6px 0 0 0;font-size:13px;opacity:.9;">Chọn vùng miền bên dưới để lọc và cập nhật biểu đồ tự động.</p>
</div>''')

instruction = widgets.HTML(value='<p style="font-size:13px;color:#555;">👇 <b>Chọn vùng miền để lọc dữ liệu:</b></p>')

with output:
    build_dashboard('Tất cả').show()

display(header, instruction, dropdown, output)
print('✅ Task 4.2 hoàn thành – Mini Interactive Dashboard')

---
## Tóm tắt kết quả

| Task | Nội dung | Trạng thái |
|------|----------|------------|
| **Bước 1** | DataFrame 10 tỉnh/thành phố: KinhDo, ViDo, DoanhThu, SoChiNhanh, VungMien | ✅ Hoàn thành |
| **Task 4.1** | Geospatial Bubble Map (`scatter_geo`) – kích thước=DoanhThu, màu=VungMien | ✅ Hoàn thành |
| **Task 4.2** | Mini Interactive Dashboard: Bản đồ + Biểu đồ Cột + Dropdown lọc vùng miền | ✅ Hoàn thành |

**Thư viện:** `pandas`, `plotly`, `ipywidgets`